# Causally informed AE

The causally informed autoencoder adds a causal discovery-based penalty (using DirectLiNGAM) on the latent space during training so the autoencoder learns representations with causal structure.

*   The causal loss computed using DirectLiNGAM. The latent space Z is fed
into DirectLiNGAM. This produces a causal adjacency matrix A.
*   A penalty is added to the AE loss:
sparsity penalty → penalizes dense causal graphs
dagness penalty → penalizes cycles (non-DAG structure)
*   Training alternates between AE reconstruction and causal penalty

→ So the AE is nudged to produce a latent space that LiNGAM sees as a sparse causal DAG.

In [ ]:
# Ensure PyTorch is installed
try:
    import torch
    import torch.nn as nn
except ImportError:
    print("PyTorch not found. Installing PyTorch...")
    !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
    import torch
    import torch.nn as nn

# Redefine the Autoencoder model using PyTorch
class Autoencoder(nn.Module):
    def __init__(self, input_dim, latent_dim, activation='relu'):
        super(Autoencoder, self).__init__()
        self.input_dim = input_dim
        self.latent_dim = latent_dim

        # Define activation function based on string input
        if activation == 'relu':
            self.activation = nn.ReLU()
        elif activation == 'tanh':
            self.activation = nn.Tanh()
        elif activation == 'leakyrelu':
            self.activation = nn.LeakyReLU(negative_slope=0.01)
        elif activation == 'sigmoid':
            self.activation = nn.Sigmoid()
        elif activation == 'linear':
             self.activation = lambda x: x # Linear activation
        else:
            raise ValueError(f"Unknown activation function: {activation}")


        # Encoder layers (PyTorch)
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 128),
            self.activation,
            nn.Linear(128, 64),
            self.activation,
            nn.Linear(64, latent_dim)
            #self.activation # Apply activation to latent space as well
        )

        # Decoder layers (PyTorch)
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            self.activation,
            nn.Linear(64, 128),
            self.activation,
            nn.Linear(128, input_dim),
            nn.Tanh()  # Output activation (usually sigmoid or tanh for normalized data)
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded, encoded # Return both reconstructed and latent representation

# The compute_causal_loss function is already compatible with PyTorch latent tensor
def compute_causal_loss(Z_latent, model_type="direct_lingam"):
    # Z_latent is a PyTorch tensor here
    Z_np = Z_latent.detach().cpu().numpy()

    # Run DirectLiNGAM (expects numpy array)
    model = DirectLiNGAM()
    model.fit(Z_np)
    A = model.adjacency_matrix_

    # Sparsity penalty (more edges = higher loss)
    sparsity_penalty = np.sum(np.abs(A)) / A.size

    # DAG-ness penalty (penalize cycles)
    # Create graph only for significant edges
    G = nx.from_numpy_array((np.abs(A) > 1e-3).astype(int), create_using=nx.DiGraph)
    try:
        # Check for cycles
        nx.find_cycle(G, orientation="original")
        dagness_penalty = 1.0
    except nx.NetworkXNoCycle:
        # No cycle found
        dagness_penalty = 0.0
    except Exception as e:
         # Handle other potential errors in graph creation/cycle detection
         print(f"Error in DAG check: {e}")
         dagness_penalty = 1.0 # Penalize if graph ops fail


    total_causal_penalty = sparsity_penalty + dagness_penalty
    # Return a PyTorch tensor
    return torch.tensor(total_causal_penalty, dtype=torch.float32)


# The train_causal_ae function (already using PyTorch)
def train_causal_ae(model, X, epochs=100, lr=1e-3, lambda_causal=1.0, causal_every=5):
    # X is a pandas DataFrame or numpy array, convert to PyTorch tensor
    X_tensor = torch.tensor(X.values if isinstance(X, pd.DataFrame) else X, dtype=torch.float32)

    model.train() # Set model to training mode
    optimizer = torch.optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    for epoch in range(epochs):
        optimizer.zero_grad()
        # Autoencoder forward pass returns reconstruction and latent
        recon, Z = model(X_tensor)
        loss_recon = criterion(recon, X_tensor)

        # Causal penalty every few epochs
        if epoch % causal_every == 0:
            # Pass the PyTorch latent tensor to causal loss
            loss_causal = compute_causal_loss(Z)
        else:
            loss_causal = torch.tensor(0.0, device=X_tensor.device) # Ensure tensor is on the same device

        # Total loss
        loss_total = loss_recon + lambda_causal * loss_causal
        loss_total.backward()
        optimizer.step()

        if epoch % 10 == 0 or epoch == epochs - 1:
            print(f"[Epoch {epoch+1}] MSE: {loss_recon.item():.4f} | Causal: {loss_causal.item():.4f} | Total: {loss_total.item():.4f}")

    model.eval()
    return model

input_dim = norm_df.shape[1]
model = Autoencoder(input_dim=input_dim, latent_dim=100, activation='leakyrelu')

# Train the model using the PyTorch training function
# Pass norm_df (pandas DataFrame or numpy array)
model = train_causal_ae(model, norm_df, epochs=100, lambda_causal=5.0)

print("\nTraining of Causal Autoencoder completed.")

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 150 iterations, alpha=4.890e-05, previous alpha=4.807e-05, with an active set of 63 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 109 iterations, alpha=1.567e-07, previous alpha=1.421e-07, with an active set of 64 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 105 iterations, alpha=1.760e-07, previous alpha=1.504e-07, with an active set of 64 regressors.
  warnings.warn(
/usr

[Epoch 1] MSE: 0.6101 | Causal: 0.1080 | Total: 1.1502


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 112 iterations, alpha=9.966e-06, previous alpha=9.848e-06, with an active set of 57 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 137 iterations, alpha=8.877e-05, previous alpha=8.681e-05, with an active set of 58 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 85 iterations, alpha=2.130e-06, previous alpha=2.093e-06, with an active set of 56 regressors.
  warnings.warn(
/usr/

[Epoch 11] MSE: 0.5597 | Causal: 0.1076 | Total: 1.0976


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 112 iterations, alpha=2.213e-06, previous alpha=2.178e-06, with an active set of 57 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 67 iterations, alpha=1.236e-04, previous alpha=1.235e-04, with an active set of 46 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 101 iterations, alpha=3.327e-05, previous alpha=3.304e-05, with an active set of 54 regressors.
  warnings.warn(
/usr/

...

[Epoch 81] MSE: 0.2940 | Causal: 0.1246 | Total: 0.9168


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 94 iterations, alpha=8.771e-05, previous alpha=8.764e-05, with an active set of 53 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 117 iterations, alpha=1.308e-05, previous alpha=1.280e-05, with an active set of 60 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 154 iterations, alpha=4.619e-06, previous alpha=4.617e-06, with an active set of 61 regressors.
  warnings.warn(
/usr/

[Epoch 91] MSE: 0.2819 | Causal: 0.1193 | Total: 0.8782


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 107 iterations, alpha=3.056e-07, previous alpha=3.009e-07, with an active set of 62 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 162 iterations, alpha=8.941e-06, previous alpha=8.920e-06, with an active set of 59 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 132 iterations, alpha=1.625e-07, previous alpha=1.563e-07, with an active set of 63 regressors.
  warnings.warn(
/usr

[Epoch 100] MSE: 0.2732 | Causal: 0.0000 | Total: 0.2732

Training of Causal Autoencoder completed.


In [ ]:
input_dim1 = norm_df1.shape[1]
model1 = Autoencoder(input_dim=input_dim1, latent_dim=100, activation='leakyrelu')

model1 = train_causal_ae(model1, norm_df1, epochs=100, lambda_causal=5.0)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 75 iterations, alpha=1.016e-06, previous alpha=1.013e-06, with an active set of 44 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 87 iterations, alpha=5.216e-07, previous alpha=5.033e-07, with an active set of 46 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 99 iterations, alpha=7.444e-06, previous alpha=7.439e-06, with an active set of 44 regressors.
  warnings.warn(
/usr/lo

[Epoch 1] MSE: 0.4214 | Causal: 0.1132 | Total: 0.9876


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 31 iterations, alpha=5.832e-04, previous alpha=5.654e-04, with an active set of 16 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 24 iterations, alpha=2.529e-07, previous alpha=2.383e-07, with an active set of 19 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 27 iterations, i.e. alpha=1.656e-07, with an active set of 19 regressors, and the smallest cholesky pivot element being 2.220e-16. Reduce max_iter or increase eps paramete

[Epoch 11] MSE: 0.1896 | Causal: 0.0587 | Total: 0.4831


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 17 iterations, alpha=2.780e-07, previous alpha=2.483e-07, with an active set of 14 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 27 iterations, alpha=4.170e-07, previous alpha=3.205e-07, with an active set of 16 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 23 iterations, alpha=3.919e-07, previous alpha=3.398e-07, with an active set of 16 regressors.
  warnings.warn(
/usr/lo

...

[Epoch 81] MSE: 0.0122 | Causal: 0.0464 | Total: 0.2442


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 22 iterations, alpha=1.236e-06, previous alpha=1.126e-06, with an active set of 13 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 25 iterations, alpha=1.915e-06, previous alpha=1.859e-06, with an active set of 14 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 31 iterations, alpha=2.965e-06, previous alpha=2.651e-06, with an active set of 16 regressors.
  warnings.warn(
/usr/lo

[Epoch 91] MSE: 0.0121 | Causal: 0.0482 | Total: 0.2531


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 25 iterations, alpha=3.793e-06, previous alpha=3.517e-06, with an active set of 12 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 24 iterations, alpha=1.035e-06, previous alpha=9.838e-07, with an active set of 13 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 34 iterations, alpha=3.398e-07, previous alpha=1.593e-07, with an active set of 19 regressors.
  warnings.warn(
/usr/lo

[Epoch 100] MSE: 0.0119 | Causal: 0.0000 | Total: 0.0119


In [ ]:
input_dim2 = norm_df2.shape[1]
model2 = Autoencoder(input_dim=input_dim2, latent_dim=50, activation='leakyrelu')

model2 = train_causal_ae(model2, norm_df2, epochs=100, lambda_causal=5.0)

[Epoch 1] MSE: 0.9780 | Causal: 0.0847 | Total: 1.4013
[Epoch 11] MSE: 0.9251 | Causal: 0.0970 | Total: 1.4101


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 26 iterations, alpha=3.108e-04, previous alpha=3.108e-04, with an active set of 21 regressors.
  warnings.warn(


[Epoch 21] MSE: 0.7499 | Causal: 0.1015 | Total: 1.2573
[Epoch 31] MSE: 0.6245 | Causal: 0.1007 | Total: 1.1282
[Epoch 41] MSE: 0.5798 | Causal: 0.1087 | Total: 1.1233
[Epoch 51] MSE: 0.5493 | Causal: 0.1062 | Total: 1.0806


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 43 iterations, alpha=6.111e-05, previous alpha=6.102e-05, with an active set of 30 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 15 iterations, alpha=1.386e-03, previous alpha=1.386e-03, with an active set of 12 regressors.
  warnings.warn(


[Epoch 61] MSE: 0.5208 | Causal: 0.1120 | Total: 1.0805
[Epoch 71] MSE: 0.4975 | Causal: 0.1150 | Total: 1.0723


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 40 iterations, alpha=1.801e-04, previous alpha=1.735e-04, with an active set of 27 regressors.
  warnings.warn(


[Epoch 81] MSE: 0.4804 | Causal: 0.1079 | Total: 1.0196
[Epoch 91] MSE: 0.4674 | Causal: 0.1220 | Total: 1.0774
[Epoch 100] MSE: 0.4565 | Causal: 0.0000 | Total: 0.4565


In [ ]:
input_dim3 = norm_df3.shape[1]
model3 = Autoencoder(input_dim=input_dim3, latent_dim=100, activation='leakyrelu')

model3 = train_causal_ae(model3, norm_df3, epochs=100, lambda_causal=5.0)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 151 iterations, alpha=3.939e-04, previous alpha=3.919e-04, with an active set of 60 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 146 iterations, alpha=4.868e-06, previous alpha=4.328e-06, with an active set of 63 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 108 iterations, alpha=3.881e-07, previous alpha=3.876e-07, with an active set of 63 regressors.
  warnings.warn(
/usr

[Epoch 1] MSE: 0.1288 | Causal: 0.1172 | Total: 0.7150


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 117 iterations, alpha=1.501e-06, previous alpha=1.265e-06, with an active set of 58 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 95 iterations, alpha=5.804e-06, previous alpha=5.738e-06, with an active set of 58 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 90 iterations, alpha=4.275e-06, previous alpha=4.185e-06, with an active set of 51 regressors.
  warnings.warn(
/usr/l

[Epoch 11] MSE: 0.1179 | Causal: 0.1017 | Total: 0.6262


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 60 iterations, alpha=1.031e-05, previous alpha=1.016e-05, with an active set of 43 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 51 iterations, alpha=3.833e-04, previous alpha=3.814e-04, with an active set of 30 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 61 iterations, alpha=1.093e-04, previous alpha=1.081e-04, with an active set of 38 regressors.
  warnings.warn(
/usr/lo

...

[Epoch 81] MSE: 0.0667 | Causal: 0.1140 | Total: 0.6365


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 97 iterations, alpha=3.643e-05, previous alpha=3.642e-05, with an active set of 52 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 140 iterations, alpha=1.966e-06, previous alpha=1.488e-06, with an active set of 57 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 116 iterations, alpha=3.140e-07, previous alpha=2.296e-07, with an active set of 57 regressors.
  warnings.warn(
/usr/

[Epoch 91] MSE: 0.0621 | Causal: 0.1154 | Total: 0.6390


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 92 iterations, alpha=4.926e-05, previous alpha=4.921e-05, with an active set of 47 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 130 iterations, alpha=9.962e-06, previous alpha=9.669e-06, with an active set of 55 regressors.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 112 iterations, alpha=8.269e-06, previous alpha=6.107e-06, with an active set of 55 regressors.
  warnings.warn(
/usr/

[Epoch 100] MSE: 0.0584 | Causal: 0.0000 | Total: 0.0584


In [ ]:
def get_latent_representation(model, data):
    model.eval()
    X_tensor = torch.tensor(data.values if isinstance(data, pd.DataFrame) else data, dtype=torch.float32)
    with torch.no_grad():
        latent = model.encoder(X_tensor)
    return latent.numpy()

reduced_data = get_latent_representation(model, norm_df)
print("Reduced data shape:", reduced_data.shape)

Reduced data shape: (493, 100)


In [ ]:
from sklearn.metrics import mean_squared_error
import torch

# Convert the pandas DataFrame norm_df to a PyTorch tensor
norm_df_tensor = torch.tensor(norm_df.values, dtype=torch.float32)


model.eval()
with torch.no_grad():
    reconstructed_data_tensor, _ = model(norm_df_tensor)
    reconstructed_data = reconstructed_data_tensor.numpy()

mse = mean_squared_error(norm_df.values, reconstructed_data)

print(f"Mean Squared Error: {mse}")

In [ ]:
from sklearn.metrics import r2_score
r2 = r2_score(norm_df, reconstructed_data)
print("R-squared:", r2)

In [ ]:
reduced_data1 = get_latent_representation(model1, norm_df1)
print("Reduced data shape:", reduced_data1.shape)

Reduced data shape: (494, 100)


In [ ]:
from sklearn.metrics import mean_squared_error
import torch

# Convert the pandas DataFrame norm_df to a PyTorch tensor
norm_df_tensor1 = torch.tensor(norm_df1.values, dtype=torch.float32)


model1.eval()
with torch.no_grad():
    reconstructed_data_tensor1, _ = model1(norm_df_tensor1)
    reconstructed_data1 = reconstructed_data_tensor1.numpy()

mse1 = mean_squared_error(norm_df1.values, reconstructed_data1)

print(f"Mean Squared Error: {mse1}")

Mean Squared Error: 0.011936427528106202


In [ ]:
from sklearn.metrics import r2_score
r2 = r2_score(norm_df1, reconstructed_data1)
print("R-squared:", r2)

In [ ]:
reduced_data2 = get_latent_representation(model2, norm_df2)
print("Reduced data shape:", reduced_data2.shape)

Reduced data shape: (350, 50)


In [ ]:
from sklearn.metrics import mean_squared_error
import torch

# Convert the pandas DataFrame norm_df to a PyTorch tensor
norm_df_tensor2 = torch.tensor(norm_df2.values, dtype=torch.float32)

model2.eval()
with torch.no_grad():
    reconstructed_data_tensor2, _ = model2(norm_df_tensor2)
    reconstructed_data2 = reconstructed_data_tensor2.numpy()

mse2 = mean_squared_error(norm_df2.values, reconstructed_data2)

print(f"Mean Squared Error: {mse2}")

Mean Squared Error: 0.45524955441449366


In [ ]:
from sklearn.metrics import r2_score
r2 = r2_score(norm_df2, reconstructed_data2)
print("R-squared:", r2)

R-squared: 0.5196060538291931


In [ ]:
reduced_data3 = get_latent_representation(model3, norm_df3)
print("Reduced data shape:", reduced_data3.shape)

Reduced data shape: (489, 100)


In [ ]:
from sklearn.metrics import mean_squared_error
import torch

# Convert the pandas DataFrame norm_df to a PyTorch tensor
norm_df_tensor3 = torch.tensor(norm_df3.values, dtype=torch.float32)

model3.eval()
with torch.no_grad():
    reconstructed_data_tensor3, _ = model3(norm_df_tensor3)
    reconstructed_data3 = reconstructed_data_tensor3.numpy()

mse3 = mean_squared_error(norm_df3.values, reconstructed_data3)

print(f"Mean Squared Error: {mse3}")

In [ ]:
from sklearn.metrics import r2_score
r2 = r2_score(norm_df3, reconstructed_data3)
print("R-squared:", r2)